In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, SRC_DIR

import pandas as pd
import os
import re
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
import numpy as np
import osmnx as ox

from src.cctv_dataset import build_dataset

In [ ]:
output = DATA_DIR / 'interim/cctv_interim.csv'
df = build_dataset()#csv_path= output
output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output, index=False)
print('Saved:', output, '| shape:', df.shape)

Date range: 2026-08-25 to 2026-08-31
ERA5 locations: 5 for 79 cameras
OSM areas: 48 for 79 cameras
1/48: Processing OSM area (13.70, 100.49)
Checkpoint saved: C:\Users\thunk\OneDrive\Documents\My_Project\Python\2110531_DSDE_2026s1_final_exam_project\data\processed\cctv_processed.csv
1/48: Finished in 1.41s | 1/1 cameras
2/48: Processing OSM area (13.70, 100.54)
Checkpoint saved: C:\Users\thunk\OneDrive\Documents\My_Project\Python\2110531_DSDE_2026s1_final_exam_project\data\processed\cctv_processed.csv
2/48: Finished in 6.30s | 2/2 cameras
3/48: Processing OSM area (13.70, 100.55)
Checkpoint saved: C:\Users\thunk\OneDrive\Documents\My_Project\Python\2110531_DSDE_2026s1_final_exam_project\data\processed\cctv_processed.csv
3/48: Finished in 6.21s | 1/1 cameras
4/48: Processing OSM area (13.71, 100.48)
Checkpoint saved: C:\Users\thunk\OneDrive\Documents\My_Project\Python\2110531_DSDE_2026s1_final_exam_project\data\processed\cctv_processed.csv
4/48: Finished in 6.20s | 1/1 cameras
5/48: Pro

In [ ]:
df = pd.read_csv(DATA_DIR / 'processed/cctv_processed.csv')

df

,camera_id,image_path,timestamp,location,location_detail,location_en,location_detail_en,latitude,longitude,ip_address,...,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,surface_pressure,road_type,lane_count,speed_limit_raw,oneway_feature,bridge_feature
0,1028,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-25 06:10:12,แยกสะพานแดง,แยกสะพานแดง,Red Bridge Intersection,Red Bridge Intersection,13.79343,100.52996,10.102.128.34,...,27.2,78,0.0,9.1,1007.3,NaN,NaN,NaN,NaN,NaN
1,1028,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-25 06:21:00,แยกสะพานแดง,แยกสะพานแดง,Red Bridge Intersection,Red Bridge Intersection,13.79343,100.52996,10.102.128.34,...,27.2,78,0.0,9.1,1007.3,NaN,NaN,NaN,NaN,NaN
2,1028,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-25 06:31:15,แยกสะพานแดง,แยกสะพานแดง,Red Bridge Intersection,Red Bridge Intersection,13.79343,100.52996,10.102.128.34,...,27.2,78,0.0,9.1,1007.3,NaN,NaN,NaN,NaN,NaN
3,1028,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-25 06:40:54,แยกสะพานแดง,แยกสะพานแดง,Red Bridge Intersection,Red Bridge Intersection,13.79343,100.52996,10.102.128.34,...,27.2,78,0.0,9.1,1007.3,NaN,NaN,NaN,NaN,NaN
4,1028,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-25 06:50:53,แยกสะพานแดง,แยกสะพานแดง,Red Bridge Intersection,Red Bridge Intersection,13.79343,100.52996,10.102.128.34,...,27.2,78,0.0,9.1,1007.3,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61314,939,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-31 20:15:07,TF-KP-04-แยกมหาดไทย,แยกมหาดไทย,connect BMAcctv,connect BMAcctv,13.76061,100.62467,10.156.102.39,...,27.3,89,0.5,4.5,1008.8,primary,NaN,NaN,True,0.0
61315,939,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-31 20:25:06,TF-KP-04-แยกมหาดไทย,แยกมหาดไทย,connect BMAcctv,connect BMAcctv,13.76061,100.62467,10.156.102.39,...,27.3,89,0.5,4.5,1008.8,primary,NaN,NaN,True,0.0
61316,939,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-31 20:35:06,TF-KP-04-แยกมหาดไทย,แยกมหาดไทย,connect BMAcctv,connect BMAcctv,13.76061,100.62467,10.156.102.39,...,27.3,89,0.5,4.5,1008.8,primary,NaN,NaN,True,0.0
61317,939,C:\Users\thunk\OneDrive\Documents\My_Project\P...,2026-08-31 20:45:06,TF-KP-04-แยกมหาดไทย,แยกมหาดไทย,connect BMAcctv,connect BMAcctv,13.76061,100.62467,10.156.102.39,...,27.3,89,0.5,4.5,1008.8,primary,NaN,NaN,True,0.0


In [ ]:
list(df["location_detail"].unique())

['แยกสะพานแดง',
 'รถมุ่งหน้าไปแยกหลานหลวง',
 'รถมุ่งหน้าไปถนนพระรามที่1',
 'รถมาจากแยกวัดเบญจมบพตร',
 'รถมาจากกำแพงเพชร-มุ่งหน้าไป',
 'รถมาจากแยกลาดพร้าว',
 'รถมาจากแยกตากสิน-มุ่งหน้าไป',
 'รถมาจากแยกคลองสาน-มุ่งหน้าไ',
 'รถมาจากอนุสาวรีย์ชัยสมรภูม',
 'รถมาจากแยกท่าพระ',
 'รถมาจากแยกรัชดา---สุทธิสาร',
 'รถมุ่งหน้าไปแยกห้วยขวาง',
 'รถมาจากแยกเจริญนครใต้',
 'รถมาจากแยกสุทธิสาร',
 'จุดตัดถนนพุทธมณฑลสาย2-ฝั่งขา',
 'รถมาจากแยกห้วยขวาง',
 'รถมาจากถนนเจริญนคร',
 'รถมาจากซอยลาดพร้าว-64',
 'รถมาจากแยกรัชดา-สุทธิสาร',
 'รถมาจากแยกโชคชัย-4',
 'รถมาจากแยกรัชโยธินมุ่งหน้า',
 'รถมาจากแยกรัชโยธิน-มุ่งหน้า',
 'รถมาจากแยกรัชดา-ลาดพร้าว-มุ่',
 'หน้ารถมาจากแยกรัชโยธิน-มุ่ง',
 'รถมาจากแยกลาดพร้าว-มุ่งหน้า',
 'ด้านขวารถมาจากแยกบางขุนพรห',
 'แยกสามเสน',
 'แยกพาณิชยการ-ด้านถนนพระรามท',
 'รถมุ่งไปหน้าไปเทียมร่วมมิต',
 'รถมาจากห้าแยกลาดพร้าว-ขาออก',
 'รถมาจากแยกอารีย์สัมพันธ์-มุ',
 'ถนนเจริญนคร-ขาเข้า',
 'แยกแม้นศรี',
 'รถมาจากสะพานพระราม-7-มุ่งหน้',
 'รถมาจากแยกสะพานควาย-มุ่งหน้',
 'รถมุ่งหน้าไปแยกเกียกกาย',


In [ ]:
df.columns

Index(['camera_id', 'image_path', 'timestamp', 'location', 'location_detail',
       'location_en', 'location_detail_en', 'latitude', 'longitude',
       'ip_address', 'image_count', 'date', 'accident_count', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'wind_speed_10m',
       'surface_pressure', 'road_type', 'lane_count', 'speed_limit_raw',
       'oneway_feature', 'bridge_feature'],
      dtype='str')